In [9]:
import pandas as pd
import sqlite3, json, re
from openai import OpenAI  # Use OpenAI library for OpenRouter

# ── STEP 0: INITIALIZE OPENROUTER ───────────────────────
# Get your key from OpenRouter Settings
from google.colab import userdata
userdata.get('OPENROUTER_API_KEY')

client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=OPENROUTER_API_KEY,
)

# ── STEP 1: LOAD DATA ────────────────────────────────
# Use 'on_bad_lines' to handle the CSV issues
df = pd.read_csv('/content/orders.csv', quotechar='"', on_bad_lines='warn')

# ── STEP 2: LAYER 1 — RULE CHECKS ────────────────────
def check_rules(row):
    issues = []
    # Fill NaNs to avoid errors in logic
    name = str(row['customer_name']) if pd.notnull(row['customer_name']) else ""
    email = str(row['email'])

    if name == "" or name.lower() == 'null': issues.append('null name')
    if not re.match(r'.+@.+\..+', email): issues.append('bad email')
    if not (18 <= row['age'] <= 120): issues.append('age invalid')
    if row['price'] <= 0: issues.append('negative price')
    if row['quantity'] > 100: issues.append('qty too high')
    return issues

df['rule_issues'] = df.apply(check_rules, axis=1)
df['rule_status'] = df['rule_issues'].apply(lambda x: 'fail' if x else 'pass')

# ── STEP 3: LAYER 2 — LLM CHECKS (OpenRouter Free) ───
suspicious = df[
    (df['rule_status'] == 'pass') &
    (df['notes'].fillna('').str.len() > 5)
]

def llm_validate(row):
    try:
        response = client.chat.completions.create(
            # Using the free coding model
            model="qwen/qwen3-coder:free",
            messages=[
                {"role": "system", "content": "You are a data quality validator. Return ONLY raw JSON."},
                {"role": "user", "content": f"Validate this order: product={row['product']}, price={row['price']}, notes='{row['notes']}'. Return JSON: {{\"verdict\":\"pass|warning|reject\",\"reason\":\"...\"}}"}
            ],
            # This helps ensure the model outputs JSON
            response_format={"type": "json_object"}
        )

        # OpenRouter/OpenAI response format is slightly different than Anthropic
        res_text = response.choices[0].message.content
        return json.loads(res_text)
    except Exception as e:
        return {"verdict": "error", "reason": str(e)}

# Applying the LLM check only to the subset
if not suspicious.empty:
    df.loc[suspicious.index, 'llm_result'] = suspicious.apply(llm_validate, axis=1)

# ── STEP 4: FINAL VERDICT ────────────────────────────
def final_verdict(row):
    if row['rule_status'] == 'fail': return 'rejected_rules'

    # Check if llm_result exists and is a dictionary
    res = row.get('llm_result')
    if isinstance(res, dict):
        return res.get('verdict', 'pass')
    return 'pass'

df['final'] = df.apply(final_verdict, axis=1)

# ── STEP 5: STORE RESULTS ────────────────────────────
conn = sqlite3.connect('pipeline.db')
# Convert the 'issues' list to a string so SQL can store it
df['rule_issues'] = df['rule_issues'].astype(str)
# Convert 'llm_result' dict to string for storage
df['llm_result'] = df['llm_result'].astype(str)

df[df['final']=='pass'].to_sql('clean_orders', conn, if_exists='replace')
df[df['final']!='pass'].to_sql('quarantine', conn, if_exists='replace')

print(f"Data processing complete. Pass rate: {(df['final']=='pass').mean():.1%}")

/tmp/ipykernel_16411/340632592.py:17: ParserWarning: Skipping line 4: expected 8 fields, saw 9
Skipping line 5: expected 8 fields, saw 9
Skipping line 6: expected 8 fields, saw 10
Skipping line 7: expected 8 fields, saw 9
Skipping line 9: expected 8 fields, saw 9
Skipping line 11: expected 8 fields, saw 9

  df = pd.read_csv('/content/orders.csv', quotechar='"', on_bad_lines='warn')


Data processing complete. Pass rate: 0.0%
